# Fine-Tuning Pipeline Smoke Test

This notebook walks through the same checks as `scripts/test_finetuning.py`: load a model, load a small pose-to-text dataset, run one short training epoch, plot the loss curve, and verify that a checkpoint was saved.

Expected outcome: every validation cell prints a `PASS` line, the loss plot renders, and `checkpoints/gemma_asl_notebook/smoke-checkpoint/trainer_state.pt` exists.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import torch
from torch.optim import AdamW
from torch.utils.data import DataLoader

from scripts.test_finetuning import create_synthetic_dataset, load_model
from src.data.pose_to_text_dataset import PoseToTextDataset, collate_pose_text_batch
from src.models.gemma_finetune import (
    FineTuneConfig,
    TrainingHistory,
    _prepare_model_inputs,
    _safe_model_forward,
    build_cosine_warmup_scheduler,
    save_training_checkpoint,
)

OUTPUT_DIR = Path('checkpoints/gemma_asl_notebook')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MAX_SAMPLES = 8
BATCH_SIZE = 2
USE_MOCK_MODEL = True  # Set to False to load real Gemma through Unsloth.

## 1. Load Model

Expected: the cell loads a tokenizer and a model with trainable parameters. Keep `USE_MOCK_MODEL=True` for a CPU-safe walkthrough; set it to `False` for the real Gemma smoke test.

In [ ]:
model, tokenizer = load_model(USE_MOCK_MODEL)
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'PASS model loaded: {type(model).__name__}, trainable_params={trainable_params:,}')
assert trainable_params > 0

## 2. Load Dataset

Expected: if processed WLASL files are absent, this creates a small synthetic pose manifest and still exercises `PoseToTextDataset` and `collate_pose_text_batch`.

In [ ]:
manifest = Path('data/processed/training_pairs/train.csv')
pose_root = None
if not manifest.exists():
    manifest, pose_root = create_synthetic_dataset(OUTPUT_DIR / 'synthetic_data', MAX_SAMPLES)

dataset = PoseToTextDataset.from_csv(manifest, pose_root=pose_root, include_face=False, normalize=True)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_pose_text_batch)
batch = next(iter(loader))
print(f"PASS dataset loaded: samples={len(dataset)}, pose_batch={tuple(batch['pose_features'].shape)}, labels={batch['texts'][:3]}")
assert 'pose_features' in batch and 'texts' in batch

## 3. Forward Pass

Expected: model output exposes both `loss` and `logits`. The current Gemma path may ignore pose tensors unless a pose adapter is attached, but the batch still verifies data collation and tokenization.

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
model.train()
inputs = _prepare_model_inputs(batch, tokenizer, device)
outputs = _safe_model_forward(model, inputs)
print(f'PASS forward pass: loss={outputs.loss.item():.4f}, logits_shape={tuple(outputs.logits.shape)}')
assert outputs.loss is not None and outputs.logits is not None

## 4. One-Epoch Training Loop

Expected: the loop completes quickly and records one loss value per batch.

In [ ]:
config = FineTuneConfig(output_dir=OUTPUT_DIR, learning_rate=2e-4, warmup_steps=0, num_epochs=1, device=str(device))
optimizer = AdamW((p for p in model.parameters() if p.requires_grad), lr=config.learning_rate)
scheduler = build_cosine_warmup_scheduler(optimizer, warmup_steps=0, total_steps=max(1, len(loader)))
losses = []

for step, raw_batch in enumerate(loader, start=1):
    optimizer.zero_grad(set_to_none=True)
    inputs = _prepare_model_inputs(dict(raw_batch), tokenizer, device)
    outputs = _safe_model_forward(model, inputs)
    outputs.loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), config.max_grad_norm)
    optimizer.step()
    scheduler.step()
    losses.append(float(outputs.loss.detach().cpu()))
    print(f'step={step} loss={losses[-1]:.4f} lr={scheduler.get_last_lr()[0]:.6g}')

print(f'PASS training loop: steps={len(losses)}, first_loss={losses[0]:.4f}, last_loss={losses[-1]:.4f}')
assert losses

## 5. Loss Curve

Expected: on the mock model, the loss usually trends down. With real Gemma and very few batches, use this as a sanity signal rather than a quality metric.

In [ ]:
plt.figure(figsize=(6, 3))
plt.plot(range(1, len(losses) + 1), losses, marker='o')
plt.xlabel('Step')
plt.ylabel('Training loss')
plt.title('Fine-tuning smoke loss curve')
plt.grid(True, alpha=0.3)
plt.show()

loss_decreased = losses[-1] <= losses[0]
print(f"Actual: first={losses[0]:.4f}, last={losses[-1]:.4f}, decreased={loss_decreased}")

## 6. Save Checkpoint

Expected: model files, tokenizer files, and `trainer_state.pt` are written under the checkpoint directory.

In [ ]:
checkpoint_dir = save_training_checkpoint(
    model=model,
    tokenizer=tokenizer,
    optimizer=optimizer,
    scheduler=scheduler,
    config=config,
    epoch=0,
    global_step=len(losses),
    history=TrainingHistory(train_loss=[sum(losses) / len(losses)], learning_rates=[scheduler.get_last_lr()[0]]),
    checkpoint_name='smoke-checkpoint',
)

print(f'PASS checkpoint saved: {checkpoint_dir}')
print(sorted(path.name for path in checkpoint_dir.iterdir()))
assert (checkpoint_dir / 'trainer_state.pt').exists()